## PROYECTO: PORTAL DE TICKETS PQRS
### ETAPA: EXTRACCIÓN (CAPA BRONZE)

    **OBJETIVO:** Ingestar datos desde Azure SQL y JSON (Data Lake) hacia la capa Bronze.

In [0]:
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

# Limpiamos widgets previos para evitar conflictos
dbutils.widgets.removeAll()

In [0]:
# 1. PARÁMETROS DINÁMICOS (WIDGETS)
# Alineados al estándar del docente

dbutils.widgets.text("storage_account", "adlspqrsdev")
dbutils.widgets.text("sql_server", "sql-pqrs-dev.database.windows.net")
dbutils.widgets.text("catalogo", "catalogo_pqrs")
dbutils.widgets.text("esquema", "bronze")

storage_account = dbutils.widgets.get("storage_account")
sql_server = dbutils.widgets.get("sql_server")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

In [0]:
#2. CREDENCIALES SEGURAS DESDE KEY VAULT",
sql_password = dbutils.secrets.get(scope="access-scope-ADLS", key="sql-password")
sql_user = "admin_pqrs"
sql_database = "sqldb-pqrs-dev"

print("✅ Parámetros y credenciales listos.")

In [0]:
#============================================================
# EXTRACCIÓN 1: ARCHIVO JSON DESDE EL DATA LAKE (RAW)
#============================================================

# A. Definición de Esquema
tickets_schema = StructType(fields=[
    StructField("id_ticket", IntegerType(), False),
    StructField("id_agente", IntegerType(), True),
    StructField("fecha_creacion", DateType(), True),
    StructField("estado", StringType(), True),
    StructField("asunto", StringType(), True),
    StructField("descripcion", StringType(), True)
])

ruta_json = "abfss://raw-data@{storage_account}.dfs.core.windows.net/tickets_pqrs.json"

In [0]:
# B. Lectura aplicando el esquema
ruta_json = f"abfss://raw-data@{storage_account}.dfs.core.windows.net/tickets_pqrs.json"
tabla_destino = f"{catalogo}.{esquema}.tickets_raw"

df_tickets = spark.read \
    .schema(tickets_schema) \
    .option("multiLine", True) \
    .json(ruta_json)
    
# C. Agregamos columna de auditoría
df_tickets = df_tickets.withColumn("ingestion_date", current_timestamp())

# D. Guardado en Bronze
df_tickets.write \
    .mode("overwrite") \
    .saveAsTable(tabla_destino)

print(f"✅ Tickets guardados en {tabla_destino}")

In [0]:
#============================================================
# EXTRACCIÓN 2: TABLA DE AGENTES DESDE AZURE SQL DATABASE
#============================================================

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
    
df_agentes = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "agentes_soporte") \
    .option("user", sql_user) \
    .option("password", sql_password) \
    .load()

# Agregamos auditoría y seleccionamos/renombramos si es necesario
df_agentes = df_agentes.withColumn("ingestion_date", current_timestamp())

df_agentes.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema}.agentes_soporte_raw")

print(f"✅ Agentes guardados en {catalogo}.{esquema}.agentes_soporte_raw")